# 3장 — 눈덩이: 강화 루프와 지수 성장

웹 챕터(3장)와 **같은 예제, 같은 수치**를 파이썬으로 계산합니다:
같은 배수로 커지는 지수 성장, 두 배 시간과 70의 법칙, 선형 성장과의 비교, 로그 눈금.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch3')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 사용자가 사용자를 데려온다

사용자 100명, 매달 지금 사용자의 r(성장률)만큼 새 사용자가 들어옵니다. 매달 ×(1 + r)이므로

$$S_n = S_0 \times (1 + r)^n$$

같은 출발의 선형 성장(매달 +$S_0 \cdot r$명, 첫 달 증가량이 같음)과 비교합니다.

**그림으로 읽기**(웹 3장 1절): 인과지도로는 `사용자 수 →(+) 입소문 가입 →(+) 사용자 수` 고리 하나입니다. 한 바퀴 따라 읽으면
사용자 수 ↑ → 입소문 가입 ↑ → 사용자 수 ↑, 처음 변화가 커지니 **강화 루프(R)**(− 0개). 저량-유량도로는 사용자 수 상자에 구름에서 오는
입소문 가입 파이프 하나, 그 밸브로 사용자 수(+)와 성장률 r(작은 원, +)이 가는 화살표로 이어져 **입소문 가입 = r × 사용자 수**가 됩니다.

In [ ]:
import math

S0 = 100

def expo(r, n):
    return S0 * (1 + r) ** n

def linear(r, n):
    return S0 + S0 * r * n

r = 0.10
for n in [0, 1, 2, 3, 6, 12, 24, 36]:
    print(f'{n:2d}개월  지수 {expo(r, n):10,.1f}명   선형 {linear(r, n):6,.0f}명')

# 웹 실험실(ReinforcingLab)·본문 수치 자가검증
assert round(expo(0.10, 2)) == 121
assert round(expo(0.10, 24)) == 985 and linear(0.10, 24) == 340
assert 286 < 10 * 1.15 ** 24 < 286.5            # 함정 문제: 10명이 15%로 2년이면 약 286명(직원 300명)
print('✅ 웹과 일치: 24개월 뒤 지수 약 985명 vs 선형 340명')

## 2. 두 배 시간과 70의 법칙

$(1 + r)^n = 2$ 를 풀면 두 배 시간이 나옵니다. "몇 번 곱해야 2가 되나"를 세는 도구가 로그예요(2부 7장).

$$n_{\times 2} = \frac{\log 2}{\log(1 + r)} \approx \frac{0.693}{r} \approx \frac{70}{r(\%)}$$

In [ ]:
def doubling(r):
    return math.log(2) / math.log(1 + r)

print(' 성장률   정확한 두 배 시간   70의 법칙')
for pct in [1, 2, 5, 7, 10, 15, 20, 30]:
    d = doubling(pct / 100)
    print(f'{pct:5d}%   {d:10.2f}개월      {70 / pct:7.2f}개월')

assert abs(doubling(0.10) - 7.27) < 0.005
assert abs(doubling(0.05) - 14.21) < 0.005
assert abs(doubling(0.07) - 10.24) < 0.005
assert abs(doubling(0.30) - 2.64) < 0.005 and abs(70 / 30 - 2.33) < 0.005
# 두 배 시간은 언제 재도 같다: 7.27개월마다 ×2
d = doubling(0.10)
assert abs(expo(0.10, 2 * d) / expo(0.10, d) - 2) < 1e-9
print('✅ 두 배 시간은 늘 같고, 70 ÷ 성장률(%)로 어림된다 (성장률이 클수록 어림이 조금 짧게 나옴)')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ns = list(range(0, 37))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, log in [(ax1, False), (ax2, True)]:
    ax.plot(ns, [expo(0.10, n) for n in ns], color='#e11d48', lw=2.5, label='지수 성장 (매달 ×1.1)')
    ax.plot(ns, [linear(0.10, n) for n in ns], color='#64748b', lw=2, label='선형 성장 (매달 +10명)')
    ax.set_xlabel('개월')
    ax.grid(alpha=0.3)
    if log:
        ax.set_yscale('log')
        ax.set_title('로그 눈금: 지수 성장은 직선')
    else:
        ax.set_ylim(0, 2000)
        ax.set_title('일반 눈금: 처음엔 붙어 있다가 폭발')
    ax.legend()
ax1.set_ylabel('사용자 수(명)')
plt.tight_layout()
plt.show()

## 3. 함정 둘: 연못의 수련, 그리고 매 순간 자라기

연못의 수련이 매일 두 배로 늘어 30일째에 연못을 다 덮는다면 절반을 덮은 날은? 그리고 한 달에 한 번 곱하는 대신
매 순간 자라면($dS/dt = rS$, 답은 $S_0 e^{rt}$) 얼마나 달라질까요?

In [ ]:
cover = [2 ** (day - 30) for day in range(31)]       # 30일째 = 1 (연못 전체)
half_day = next(day for day, c in enumerate(cover) if c >= 0.5)
print(f'연못의 절반을 덮은 날: {half_day}일째')
assert half_day == 29

# 매달 한 번 곱하기 vs 한 달을 m조각으로 썰어 곱하기 → e^r 로 수렴
r, months = 0.10, 12
for m in [1, 2, 4, 30, 1000]:
    print(f'한 달을 {m:4d}조각: 12개월 뒤 {((1 + r / m) ** (m * months)):.4f}배')
print(f'매 순간 자라기(e^(r·t)): {math.exp(r * months):.4f}배')
assert abs((1 + r) ** 12 - 3.138) < 0.001 and abs(math.exp(1.2) - 3.320) < 0.001
print('✅ 칸을 잘게 썰수록 연속 성장(e)에 다가간다 (5장 오일러법)')

## 4. 오늘의 발견

- 유량이 저량에 비례하면(**강화 루프**) 저량은 같은 배수로 커진다: $S_n = S_0(1 + r)^n$.
- **두 배 시간**은 언제 재도 같고 $\log 2 / \log(1 + r) \approx 70 / r(\%)$. 10%면 약 7.3개월.
- 같은 출발이라도 2년 뒤 지수(985명)와 선형(340명)은 세 배 가까이 벌어진다. 처음엔 얌전해서 과소평가하기 쉽다.
- **로그 눈금**에서 지수 성장은 직선. (가로·세로 둘 다 로그일 때 직선이면 멱법칙, 10부.)
- 영원한 지수 성장은 없다. 한계(균형 루프)를 만나면 S자로 휜다(7장).

다음 장(4장 온도 조절기)에서 반대쪽 고리, 균형 루프를 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**